# QURIOS — IA com redes tensoriais e quantização, na GPU grátis (Kaggle ou Google Colab)

Três experiências, todas com comparação justa (mesmos dados, mesmos passos, mesma semente) e medidas num teste que nenhum método viu:

**Parte 1 — encolher um modelo pronto.** Qwen2.5-0.5B-Instruct (Apache-2.0, fala português). Contra o que o mercado usa hoje (guardar cada peso com 8, 4 ou 3 bits, arredondando), o QURIOS escolhe a escala de cada canal pelas ativações reais e soma uma correção de posto baixo do erro que sobrou. O tamanho é contado em bits de verdade. Sem treino: leva uns 15 minutos.

**Parte 2 — ajustar um modelo pronto a uma tarefa nova.** Os pesos ficam congelados e só uma correção pequena aprende. LoRA (o padrão) contra o adaptador em rede tensorial do QURIOS, contra um LoRA com o mesmo número de parâmetros do TT e contra o menor LoRA possível (posto 1). A taxa de aprendizado de cada um é escolhida numa validação separada. Cerca de 1 hora.

**Parte 3 — nascer em rede tensorial.** Um modelo de linguagem pequeno treinado do zero em cinco versões: denso, rede tensorial TT, posto baixo, denso estreito e uma TT bem mais larga (largura 768), as quatro últimas com o mesmo número de parâmetros. Cerca de 1h30.

Cada parte grava o resultado em JSON **depois de cada modelo**: se a sessão cair, o que já terminou fica salvo (e impresso na tela).

**No Kaggle** (roda nos servidores dele, mesmo com o navegador fechado):
1. Em *Settings*, à direita: **Accelerator → GPU T4 x2** e **Internet → On** (precisa confirmar o telefone na conta Kaggle).
2. **Save Version → Save & Run All (Commit)**.
3. No fim, na aba *Output* da versão: os arquivos `resultado_*.json`.

**No Google Colab** (se a fila de GPU do Kaggle estiver parada):
1. *Ambiente de execução → Alterar o tipo de ambiente de execução →* **GPU T4**.
2. *Ambiente de execução →* **Executar tudo**, e deixe a aba aberta e o computador ligado até o fim.
3. Os JSON ficam em `/content` (ícone de pasta, à esquerda).

O QURIOS é instalado direto do GitHub público (github.com/matheusStark25/quirios-engine). Nada aqui usa senha ou chave sua: os modelos e os textos são públicos.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# instala o QURIOS direto do GitHub público; se você subiu o .whl como dataset, usa ele
!WHL=$(find /kaggle/input -name "quirios_engine-*.whl" 2>/dev/null | head -n 1); if [ -n "$WHL" ]; then pip install -q "$WHL" "transformers>=4.45" datasets; else pip install -q "quirios-engine @ git+https://github.com/matheusStark25/quirios-engine.git" "transformers>=4.45" datasets; fi

In [ ]:
import gc, json, os, torch
from quirios_engine import llm_tensorial as lt, llm_nativo as ln, adaptadores as ad

# a mesma pasta de saída serve no Kaggle, no Colab ou no seu computador
if os.path.isdir("/kaggle/working"):
    PASTA = "/kaggle/working"
elif os.path.isdir("/content"):
    PASTA = "/content"
else:
    PASTA = os.getcwd()
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", dispositivo, "| resultados em:", PASTA)

def gravar(nome, res):
    """Grava o resultado (parcial ou final) e devolve o caminho: nada se perde se a sessão cair."""
    arq = os.path.join(PASTA, nome)
    with open(arq, "w", encoding="utf-8") as f:
        json.dump(res, f, ensure_ascii=False, indent=1)
    return arq

def limpar():
    gc.collect()
    if dispositivo == "cuda":
        torch.cuda.empty_cache()

## Parte 1 — quantização guiada pelos dados contra o arredondamento comum

Todas as camadas lineares dos 24 blocos do Qwen2.5-0.5B. `intN` é o arredondamento por grupos de 128 (o padrão); `+dados` escolhe a escala de cada canal pelas ativações (critério: o erro na saída da camada); `+correcao` soma uma correção de posto baixo que custa cerca de meio bit por peso. Os embeddings ficam em 16 bits em todos.

In [ ]:
METODOS_Q = ("int8", "int4", "int4+dados", "int4+dados+correcao", "int3", "int3+dados", "int3+dados+correcao")
def linha_q(r):
    l = r["linhas"][-1]
    print(f"  {l['modelo']}: {l['mb']:.0f} MB ({l['compressao']:.2f}x menor), perplexidade {l['perplexidade']:.2f}", flush=True)
    gravar("resultado_quantizacao.json", r)
res_q = lt.demo("Qwen/Qwen2.5-0.5B-Instruct", metodos=METODOS_Q, alvos=lt.ALVOS, max_tokens=32768,
                dispositivo=dispositivo, ao_linha=linha_q,
                perguntas=("Qual é a capital da França?", "Explique em uma frase o que é um computador quântico.",
                           "Dê três dicas para dormir melhor."))
print(lt.texto_demo(res_q))
print("salvo:", gravar("resultado_quantizacao.json", res_q))
limpar()

## Parte 2 — ajuste fino: adaptador em rede tensorial contra LoRA

Tarefa: conversas do dia a dia, no formato de chat do próprio modelo. Mesmos 300 passos para todos; cada método testa três taxas de aprendizado e fica com a melhor na validação; a nota final é a perplexidade no teste.

Duas perguntas: (1) com os **mesmos parâmetros**, o TT ganha do LoRA? (2) **abaixo do menor LoRA possível** (posto 1), o TT posto 4 entrega mais? Na CPU, com o SmolLM2-135M e 80 passos, a resposta da (1) foi não: o LoRA ganhou.

In [ ]:
progresso_a = lambda r, s, l, dt: print(f"  [{r}] passo {s}: perda {l:.3f} ({dt/60:.1f} min)", flush=True) if s % 100 == 0 else None
def linha_a(r):
    l = r["linhas"][-1]
    print(f"  {l['modelo']}: {l['treinaveis']:,} parâm. treináveis, perplexidade {l['ppl_teste']:.3f} (lr {l['lr']:g})", flush=True)
    gravar("resultado_adaptadores.json", r)
res_a = ad.demo("Qwen/Qwen2.5-0.5B-Instruct",
                configs=(("lora", 8), ("tt", 8), ("lora", "mesmos"), ("tt", 4), ("lora", 1)), passos=300,
                lote=4, seq=256, lrs=(1e-3, 3e-3, 1e-2), d=4, dispositivo=dispositivo,
                ao_progresso=progresso_a, ao_linha=linha_a)
print(ad.texto_demo(res_a))
print("salvo:", gravar("resultado_adaptadores.json", res_a))
limpar()

## Parte 3 — modelo nativo em rede tensorial, maior que o da CPU

In [ ]:
linhas_n = []
def mostra(l):
    linhas_n.append(l)
    print(f"{l['modelo']}: {l['parametros']:,} parâm., {l['bits_por_caractere']:.3f} bits/caractere ({l['s_treino']/60:.1f} min)", flush=True)
    gravar("resultado_nativo_parcial.json", {"linhas": linhas_n})
nat = ln.experimento(passos=5000, largura=384, camadas=6, cabecas=6, contexto=256, posto=24, lote=32,
                     dispositivo=dispositivo, ao_progresso=mostra, largura_tt_larga=768)
print(ln.texto_experimento(nat))
print("salvo:", gravar("resultado_nativo.json", nat))